# A. Implémentation de k-means séquentiel

## 1.Generation des données

In [1]:
import numpy as np
from typing import Tuple, Iterator
import csv
import matplotlib.pyplot as plt

def generate_data(n_points_per_class: int = 100) -> None:
    """
    Génère un jeu de données avec deux classes et sauvegarde dans un fichier CSV.

    Args:
        n_points_per_class: Nombre de points par classe
    """
    # Génération des points pour la classe 1 autour de (5, 5)
    class1 = np.random.normal(loc=[5, 5], scale=1.0, size=(n_points_per_class, 2))

    # Génération des points pour la classe 2 autour de (10, 10)
    class2 = np.random.normal(loc=[10, 10], scale=1.0, size=(n_points_per_class, 2))

    # Combinaison des deux classes
    data = np.vstack((class1, class2))

    # Sauvegarde dans un fichier CSV
    with open('data.csv', 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['x', 'y'])  # En-têtes
        writer.writerows(data)

## 2.Lecture des données

In [2]:
def read_data(filename: str) -> Iterator[np.ndarray]:
    """
    Lit les données depuis un fichier CSV de manière efficace en mémoire.

    Args:
        filename: Nom du fichier CSV

    Yields:
        Un point sous forme de numpy array
    """
    with open(filename, 'r') as f:
        reader = csv.reader(f)
        next(reader)  # Skip header
        for row in reader:
            yield np.array([float(row[0]), float(row[1])])


## 3.Implémenter l’algorithme k-means séquentiel

In [3]:
class KMeansSequential:
    def __init__(self, k: int):
        """
        Initialise l'algorithme k-means séquentiel.

        Args:
            k: Nombre de clusters
        """
        self.k = k
        self.centers = None
        self.counts = None

    def initialize_centers(self, first_k_points: list) -> None:
        """
        Initialise les centres avec les k premiers points.

        Args:
            first_k_points: Liste des k premiers points
        """
        self.centers = np.array(first_k_points)
        self.counts = np.ones(self.k)

    def find_nearest_center(self, point: np.ndarray) -> int:
        """
        Trouve le centre le plus proche d'un point.

        Args:
            point: Point à classifier

        Returns:
            Index du centre le plus proche
        """
        distances = np.linalg.norm(self.centers - point, axis=1)
        return np.argmin(distances)

    def update_center(self, point: np.ndarray, center_idx: int) -> None:
        """
        Met à jour un centre et son effectif.

        Args:
            point: Nouveau point
            center_idx: Index du centre à mettre à jour
        """
        self.counts[center_idx] += 1
        self.centers[center_idx] += (1 / self.counts[center_idx]) * (point - self.centers[center_idx])

    def fit(self, data_iterator: Iterator[np.ndarray]) -> None:
        """
        Entraîne le modèle sur les données.

        Args:
            data_iterator: Itérateur sur les points
        """
        # Initialisation avec les k premiers points
        first_k_points = []
        for _ in range(self.k):
            first_k_points.append(next(data_iterator))
        self.initialize_centers(first_k_points)

        # Traitement des points restants
        for point in data_iterator:
            nearest_center = self.find_nearest_center(point)
            self.update_center(point, nearest_center)

    def predict(self, data_iterator: Iterator[np.ndarray]) -> Iterator[int]:
        """
        Prédit les clusters pour de nouvelles données.

        Args:
            data_iterator: Itérateur sur les points

        Yields:
            Label du cluster pour chaque point
        """
        for point in data_iterator:
            yield self.find_nearest_center(point)

## 4.Enregistrement des fichier et validation de la coherence des resultats

In [4]:
def save_results(data_iterator: Iterator[np.ndarray], predictions: Iterator[int], filename: str) -> None:
    """
    Sauvegarde les résultats dans un fichier CSV.

    Args:
        data_iterator: Itérateur sur les points
        predictions: Itérateur sur les prédictions
        filename: Nom du fichier de sortie
    """
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['x', 'y', 'cluster'])
        for point, pred in zip(data_iterator, predictions):
            writer.writerow([point[0], point[1], pred])

def visualize_results(filename: str) -> None:
    """
    Visualise les résultats.

    Args:
        filename: Nom du fichier contenant les résultats
    """
    points = []
    clusters = []

    with open(filename, 'r') as f:
        reader = csv.reader(f)
        next(reader)  # Skip header
        for row in reader:
            points.append([float(row[0]), float(row[1])])
            clusters.append(int(row[2]))

    points = np.array(points)
    clusters = np.array(clusters)

    plt.figure(figsize=(10, 6))
    plt.scatter(points[:, 0], points[:, 1], c=clusters, cmap='viridis')
    plt.title('Résultats du clustering K-means')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.colorbar(label='Cluster')
    plt.savefig('clustering_results.png')
    plt.close()

def main():
    # 1. Génération des données
    print("Génération des données...")
    generate_data()

    # 2 & 3. Lecture des données et application de k-means
    print("Application de k-means...")
    kmeans = KMeansSequential(k=2)
    kmeans.fit(read_data('data.csv'))

    # 4. Sauvegarde des résultats
    print("Sauvegarde des résultats...")
    predictions = kmeans.predict(read_data('data.csv'))
    save_results(read_data('data.csv'), predictions, 'results.csv')

    # 5. Visualisation des résultats
    print("Visualisation des résultats...")
    visualize_results('results.csv')

    print("Terminé ! Les résultats ont été sauvegardés dans 'results.csv' et 'clustering_results.png'")

if __name__ == "__main__":
    main()

Génération des données...
Application de k-means...
Sauvegarde des résultats...
Visualisation des résultats...
Terminé ! Les résultats ont été sauvegardés dans 'results.csv' et 'clustering_results.png'


## Conclusion sur l'implémentation de k-means séquentiel (Partie A)

Cette implémentation de k-means séquentiel répond aux objectifs du projet en proposant une solution adaptée au traitement de grands volumes de données. Analysons les différents aspects de cette implémentation :

### Choix d'implémentation et gestion de la mémoire

- L'utilisation d'itérateurs (`Iterator`) et de générateurs permet une lecture point par point des données, répondant à la contrainte de consommation mémoire limitée
- L'algorithme maintient uniquement en mémoire les centres (μ) et les effectifs (n) de chaque cluster, garantissant une empreinte mémoire constante O(k) où k est le nombre de clusters
- La mise à jour incrémentale des centres permet de traiter les points sans avoir à les stocker, répondant parfaitement à la contrainte "l'exécution doit pouvoir être possible si le nombre de points augmente drastiquement"

### Analyse en termes de parallélisation

- Cette version séquentielle présente des limitations en termes de parallélisation car les points sont traités un par un
- La dépendance séquentielle dans la mise à jour des centres (chaque mise à jour dépend de l'état précédent) rend difficile la parallélisation de l'algorithme
- Ces limitations justifient l'exploration des autres approches (streaming et distribuée) dans les parties suivantes du projet

### Validation de la cohérence des résultats

- L'implémentation inclut des outils de visualisation permettant de valider graphiquement la qualité du clustering
- La sauvegarde des résultats respecte la contrainte de faible consommation mémoire grâce à l'écriture ligne par ligne
- Le jeu de données généré avec deux classes distinctes permet de vérifier visuellement la pertinence des clusters obtenus

### Limites et perspectives d'amélioration

- L'algorithme est sensible à l'ordre d'arrivée des points, ce qui sera adressé dans la partie B avec l'approche streaming
- L'absence de parallélisation limite les performances sur de très grands volumes de données, problématique qui sera traitée dans les parties C et D avec Apache Beam
- L'initialisation par les premiers points pourrait être améliorée, bien que cela ne soit pas une priorité dans le contexte de l'étude des approches de parallélisation

Cette implémentation constitue une base de référence démontrant les compromis entre gestion de mémoire et capacité de parallélisation, servant ainsi de point de comparaison pour les approches alternatives qui seront développées dans les parties suivantes du projet.

# B. Implémentation d’une version streaming de k-means

## 1.Implémentation de la class

In [5]:
import numpy as np
from sklearn.cluster import KMeans
from typing import List, Tuple, Dict
import csv
from datetime import datetime
import matplotlib.pyplot as plt
from collections import deque

class StreamingKMeans:
    def __init__(self, n_clusters: int, max_batches: int = 5, history_weight: float = 0.8):
        """
        Initialise le modèle de streaming k-means.

        Args:
            n_clusters: Nombre de clusters (k)
            max_batches: Nombre maximum de batches à conserver en mémoire (T)
            history_weight: Poids à accorder à l'historique (r)
        """
        self.n_clusters = n_clusters
        self.max_batches = max_batches
        self.history_weight = history_weight
        self.batches = deque(maxlen=max_batches)  # Utilisation de deque pour gérer automatiquement la taille max
        self.centroids = None
        self.partition = None

    def _compute_weights(self) -> np.ndarray:
        """
        Calcule les poids pour chaque point basés sur l'âge du batch.

        Returns:
            Array des poids pour chaque point
        """
        weights = []
        for batch_idx in range(len(self.batches)):
            # Le batch le plus récent a l'index 0
            batch_weight = self.history_weight ** batch_idx
            weights.extend([batch_weight] * len(self.batches[batch_idx]))
        return np.array(weights)

    def _prepare_data(self) -> np.ndarray:
        """
        Prépare toutes les données des batches en un seul array.

        Returns:
            Array combinant tous les points de tous les batches
        """
        return np.vstack(self.batches)

    def partial_fit(self, batch: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """
        Met à jour le modèle avec un nouveau batch de données.

        Args:
            batch: Nouveau batch de données

        Returns:
            Tuple contenant les centroïdes et les labels de partition
        """
        # Ajout du nouveau batch (deque gère automatiquement la suppression du plus ancien si nécessaire)
        self.batches.appendleft(batch)

        # Préparation des données et des poids
        X = self._prepare_data()
        weights = self._compute_weights()

        # Initialisation des centroïdes si c'est le premier batch
        init = self.centroids if self.centroids is not None else 'k-means++'

        # Application de k-means pondéré
        kmeans = KMeans(n_clusters=self.n_clusters, init=init, n_init=1)
        self.partition = kmeans.fit_predict(X, sample_weight=weights)
        self.centroids = kmeans.cluster_centers_

        return self.centroids, self.partition

## 2.generation de batch et visualisation des resultats

In [6]:
def generate_batch(n_points: int = 50, time_step: int = 0) -> np.ndarray:
    """
    Génère un batch de données avec drift temporel.

    Args:
        n_points: Nombre de points dans le batch
        time_step: Étape temporelle pour simuler le drift

    Returns:
        Array de points générés
    """
    # Simulation d'un concept drift en déplaçant les centres au fil du temps
    center1 = np.array([5 + time_step * 0.5, 5 + time_step * 0.3])
    center2 = np.array([10 - time_step * 0.3, 10 + time_step * 0.2])

    # Génération des points
    points1 = np.random.normal(loc=center1, scale=1.0, size=(n_points // 2, 2))
    points2 = np.random.normal(loc=center2, scale=1.0, size=(n_points // 2, 2))

    return np.vstack((points1, points2))

def visualize_streaming_results(data: np.ndarray, labels: np.ndarray,
                              centroids: np.ndarray, batch_idx: int):
    """
    Visualise les résultats du clustering pour un batch donné.

    Args:
        data: Données à visualiser
        labels: Labels des clusters
        centroids: Positions des centroïdes
        batch_idx: Index du batch pour le nom du fichier
    """
    plt.figure(figsize=(10, 6))

    # Affichage des points
    plt.scatter(data[:, 0], data[:, 1], c=labels, cmap='viridis', alpha=0.6)

    # Affichage des centroïdes
    plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='x', s=200,
               linewidths=3, label='Centroids')

    plt.title(f'Streaming K-means - Batch {batch_idx}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.savefig(f'streaming_results_batch_{batch_idx}.png')
    plt.close()

def main():
    # Paramètres
    n_clusters = 2
    n_batches = 10
    points_per_batch = 100
    max_batches_memory = 5
    history_weight = 0.8

    # Initialisation du modèle
    model = StreamingKMeans(n_clusters=n_clusters,
                           max_batches=max_batches_memory,
                           history_weight=history_weight)

    # Traitement des batches
    for i in range(n_batches):
        print(f"Traitement du batch {i}...")

        # Génération d'un nouveau batch avec drift temporel
        batch = generate_batch(n_points=points_per_batch, time_step=i)

        # Mise à jour du modèle
        centroids, partition = model.partial_fit(batch)

        # Visualisation des résultats
        # On visualise uniquement les données du batch actuel pour plus de clarté
        visualize_streaming_results(batch, partition[:len(batch)], centroids, i)

        print(f"Centroïdes actuels:\n{centroids}\n")

if __name__ == "__main__":
    main()

Traitement du batch 0...
Centroïdes actuels:
[[5.12195632 4.99723468]
 [9.91738087 9.82349718]]

Traitement du batch 1...
Centroïdes actuels:
[[5.30932211 5.27375319]
 [9.77902246 9.91345428]]

Traitement du batch 2...
Centroïdes actuels:
[[ 5.60108647  5.35771359]
 [ 9.5749521  10.15137366]]

Traitement du batch 3...
Centroïdes actuels:
[[ 5.8682319   5.46693842]
 [ 9.41482993 10.31678964]]

Traitement du batch 4...
Centroïdes actuels:
[[ 6.24084942  5.7000323 ]
 [ 9.25292164 10.49232571]]

Traitement du batch 5...
Centroïdes actuels:
[[ 6.68122623  5.87556697]
 [ 8.9205149  10.6114754 ]]

Traitement du batch 6...
Centroïdes actuels:
[[ 7.24879256  6.25380421]
 [ 8.58154226 10.88837126]]

Traitement du batch 7...
Centroïdes actuels:
[[ 7.88368726  6.5038958 ]
 [ 8.37161231 11.11429274]]

Traitement du batch 8...
Centroïdes actuels:
[[ 8.34615565  6.81191216]
 [ 8.03840711 11.29227428]]

Traitement du batch 9...
Centroïdes actuels:
[[ 8.88321398  7.13102449]
 [ 7.69133213 11.48121249]]

## Conclusion sur l'implémentation streaming de k-means (Partie B)

Cette implémentation streaming de k-means propose une approche différente pour gérer les grands volumes de données, avec un focus particulier sur l'adaptation aux changements temporels des données. Analysons les aspects clés de cette implémentation :

### Gestion de la mémoire et des données temporelles

- L'utilisation de `deque` avec une taille maximale (`max_batches`) permet de maintenir une empreinte mémoire constante
- La conservation des T derniers batches offre un compromis entre mémoire utilisée et capacité d'adaptation
- Le système de pondération avec le paramètre r (`history_weight`) permet de donner plus d'importance aux données récentes tout en conservant une influence des données historiques

### Adaptation au concept drift

- L'algorithme s'adapte naturellement aux changements de distribution des données grâce au mécanisme de fenêtre glissante
- La pondération exponentielle décroissante (`r^t`) permet une transition progressive lors de l'évolution des clusters
- L'utilisation de scikit-learn pour le clustering pondéré garantit une initialisation robuste avec k-means++

### Avantages par rapport à l'approche séquentielle (Partie A)

- Meilleure gestion des changements temporels dans les données
- Possibilité de réévaluer les affectations aux clusters sur plusieurs batches
- Moins sensible à l'ordre d'arrivée des points grâce à la conservation partielle de l'historique
- Initialisation plus robuste des centroïdes grâce à k-means++

### Limites et compromis

- Consommation mémoire plus importante que l'approche séquentielle (stockage de T batches)
- Coût computationnel plus élevé dû à la réexécution de k-means sur plusieurs batches
- Le choix des hyperparamètres (T et r) peut impacter significativement les performances
- Pas de parallélisation native, ce qui sera adressé dans les parties suivantes avec Apache Beam

Cette implémentation offre une solution élégante au problème du concept drift tout en maintenant une complexité mémoire maîtrisée, préparant ainsi le terrain pour l'exploration des approches distribuées dans les parties suivantes du projet.

# C. Implémentation de k-means distribué(Apache Beam)

In [7]:
! pip install apache_beam -q

In [8]:
import apache_beam as beam
import random
import math
from apache_beam.options.pipeline_options import PipelineOptions

## Points

In [9]:
points= [(1, 2), (3, 4), (5, 6), (7, 8), (9, 10), (11, 12), (13, 14), (15, 16)]

## PCollection

In [10]:
with beam.Pipeline() as pipeline:
    pcollections_points =(
        pipeline | "Create Points" >> beam.Create(points)
    )
    pcollections_points | "Afficher les points" >> beam.Map(print)

(1, 2)
(3, 4)
(5, 6)
(7, 8)
(9, 10)
(11, 12)
(13, 14)
(15, 16)


## Assigner chaque point à un cluster

In [11]:
# Nombre de clusters
k = 2
def assign_random_cluster(points, num_clusters):
    id_cluster= random.randint(0, num_clusters - 1)
    return id_cluster, points

with beam.Pipeline() as pipeline:
    pcollections_points =(
        pipeline | "Create Points" >> beam.Create(points)
    )
    #pcollections_points | "Afficher les points" >> beam.Map(print)
    cluster= (
        pcollections_points | "Assigner chaque points un cluster" >> beam.Map(assign_random_cluster,k )
        | "Grouper les points en fonction du cluster" >> beam.GroupByKey()
    )
    cluster | "Afficher les points avec le cluster associé" >> beam.Map(print)



(0, [(1, 2), (7, 8), (13, 14), (15, 16)])
(1, [(3, 4), (5, 6), (9, 10), (11, 12)])


## Centroides

In [12]:
def campute_centroids(points_clusters):
    clusters_id, list_points= points_clusters
    length= len(list_points)
    centroids_x= sum(x[0] for x in list_points)/length
    centroids_y= sum(y[1] for y in list_points)/length
    return clusters_id, (centroids_x, centroids_y)


with beam.Pipeline() as pipeline:
    pcollections_points =(
        pipeline | "Create Points" >> beam.Create(points)
    )
    #pcollections_points | "Afficher les points" >> beam.Map(print)

    cluster= (
        pcollections_points | "Assigner chaque points un cluster" >> beam.Map(assign_random_cluster,k )
        | "Grouper les points en fonction du cluster" >> beam.GroupByKey()
    )
    cluster | "Afficher les points avec le cluster associé" >> beam.Map(print)


    centoids_clusters= (
        cluster | " \n Centroids de chaque cluster" >> beam.Map(campute_centroids)
    )
    centoids_clusters | "Afficher les centroides de chaque cluster" >> beam.Map(print)

(1, (7.666666666666667, 8.666666666666666))
(1, [(1, 2), (3, 4), (7, 8), (9, 10), (11, 12), (15, 16)])
(0, (9.0, 10.0))
(0, [(5, 6), (13, 14)])


## Implémentation de l'étape partitionnement

### assign_cluster

In [13]:

def assign_cluster(point, centroides):
    """
    Associe un point au cluster le plus proche.

    Args:
        point (tuple): Les coordonnées du point (x, y).
        centroids (dict): Dictionnaire des centroïdes {cluster_id: (centroid_x, centroid_y)}.

    Returns:
        tuple: (cluster_id, point) où cluster_id est le numéro du cluster le plus proche.
    """
    closest_cluster = None
    min_distance=float("inf")
    for cluster_id, centoids in centroides.items():
        # Calcul de la distance euclidienne
        distance = math.sqrt( (point[0] - centoids[0])**2 + (point[1] - centoids[1])**2 )
        if distance < min_distance:
            min_distance = distance
            closest_cluster= cluster_id
    return closest_cluster, point

In [21]:
centroids = {
    0: (1, 1),
    1: (5, 5),
    2: (10, 10)
}
point = (8, 8)
result = assign_cluster(point, centroids)
print(result)


(2, (8, 8))


In [15]:
# Données de test
points= [(1,1), (2, 2), (6, 6), (7,7), (9, 9),(11,11)]
# Je donne une liste de centroids avec coordonnées

centroids = {0: (1, 1), 1: (5, 5), 2: (10, 10)}

with beam.Pipeline() as pipeline:
    pcollection_points =(
        pipeline | "Create Points" >> beam.Create(points)
    )
    #pcollection_points | "Affiches points" >> beam.Map(print)

    pcollection_centroids= (
        pipeline | "Create Centroids" >> beam.Create([centroids])
    )
    #pcollection_centroids | "afficher centroids" >> beam.Map(print)

    # Assigner à chaque point au cluster le plus proche
    assigned_points = (
        pcollection_points
        | "Assign Closest Cluster" >> beam.Map(
            assign_cluster,
            centroides=beam.pvalue.AsSingleton(pcollection_centroids)
        )
    )
    # Afficher les points assignés
    assigned_points | "Afficher cluster et point" >> beam.Map(print)

(0, (1, 1))
(0, (2, 2))
(1, (6, 6))
(1, (7, 7))
(2, (9, 9))
(2, (11, 11))


In [16]:
# Points d'entrée
points = [(1, 1), (2, 2), (4, 3), (3, 3), (6, 6), (7, 7), (9, 9), (2, 10), (10, 2), (11, 11)]
k = 2  # Nombre de clusters

# Pipeline Apache Beam
with beam.Pipeline() as pipeline:
    # 1. Créer une PCollection des points
    pcollections_points = pipeline | "Create Points" >> beam.Create(points)

    # 2. Associer chaque point à un cluster aléatoire
    clusters = (
        pcollections_points
        | "Assign Random Clusters" >> beam.Map(assign_random_cluster, k)
        | "Group by Cluster" >> beam.GroupByKey()
    )

    # 3. Calculer les centroïdes
    centroids = (
        clusters
        | "Compute Centroids" >> beam.Map(campute_centroids)
        | "Centroids to Dict" >> beam.transforms.combiners.ToDict()
    )
    # entroids | "Print centroids Points" >> beam.Map(print)
    # Assigner à chaque point au cluster le plus proche
    assigned_points = (
        pcollections_points
        | "Assign Closest Cluster" >> beam.Map(
            assign_cluster,
            centroides=beam.pvalue.AsSingleton(centroids)
        )
        | "Commbiner par cluster" >> beam.GroupByKey()
    )
    assigned_points | " Print Assigned Points" >> beam.Map(print)

(1, [(1, 1), (2, 2), (4, 3), (3, 3), (10, 2)])
(0, [(6, 6), (7, 7), (9, 9), (2, 10), (11, 11)])


### Les suppositions principales sont :

- Ils sont connus à l'avance.

- On considére qu'ils sont statiques.

- Il peuvent tenir en mémoire.

## Conclusion sur l'implémentation de k-means distribué avec Apache Beam (Partie C)

Cette implémentation distribuée de k-means avec Apache Beam présente une approche différente pour traiter les données à grande échelle. Analysons les aspects clés de cette implémentation :

### Choix d'implémentation et parallélisation

- L'utilisation de `PCollection` permet une distribution naturelle des données
- La décomposition en étapes distinctes (assignation aléatoire, calcul des centroïdes, assignation au plus proche) facilite la parallélisation
- Les transformations comme `GroupByKey` permettent de gérer efficacement les regroupements de données distribuées

### Gestion des centroïdes

- L'approche utilise une diffusion des centroïdes via `beam.pvalue.AsSingleton`
- Les centroïdes sont considérés comme statiques pendant chaque itération
- La transformation en dictionnaire des centroïdes permet un accès efficace lors de l'assignation des points

### Limites et suppositions

- Les centroïdes sont supposés pouvoir tenir en mémoire
- L'implémentation actuelle considère les centroïdes comme statiques
- La dimension des données est fixée à 2D, bien que l'extension à plus de dimensions soit possible

### Avantages par rapport aux approches précédentes

- Capacité à traiter des volumes de données beaucoup plus importants grâce à la distribution
- Parallélisation native des calculs de distance et d'assignation des points
- Scalabilité horizontale possible grâce à l'architecture Apache Beam

Cette implémentation pose les bases d'une approche distribuée, préparant le terrain pour la version séquentielle distribuée de la partie D, tout en mettant en évidence les compromis nécessaires entre distribution des données et gestion des centroïdes.

# D. Implémentation de k-means séquentiel distribuée

## Kmeans sequentiel distribué avec une seul clé

In [17]:
import apache_beam as beam
from apache_beam.transforms.userstate import BagStateSpec
import numpy as np

class KMeansSequentialState(beam.DoFn):
    # Définition des états pour stocker les centres et leurs effectifs
    centers_state = BagStateSpec('centers', beam.coders.PickleCoder())
    counts_state = BagStateSpec('counts', beam.coders.PickleCoder())

    def __init__(self, k: int):
        self.k = k

    def process(self, element, centers=beam.DoFn.StateParam(centers_state),
                counts=beam.DoFn.StateParam(counts_state)):
        # element est maintenant un tuple (key, point)
        key, point_data = element
        point = np.array([float(point_data[0]), float(point_data[1])])

        # Initialisation des centres si nécessaire
        current_centers = list(centers.read())
        if not current_centers:
            centers.clear()
            counts.clear()
            centers.add(point)
            counts.add(1)
            return

        # Trouver le centre le plus proche
        distances = [np.linalg.norm(point - center) for center in current_centers]
        nearest_center_idx = np.argmin(distances)

        # Mise à jour des compteurs
        current_counts = list(counts.read())
        new_count = current_counts[nearest_center_idx] + 1

        # Mise à jour du centre
        current_centers[nearest_center_idx] = (
            current_centers[nearest_center_idx] +
            (1.0 / new_count) * (point - current_centers[nearest_center_idx])
        )

        # Sauvegarder les nouveaux états
        centers.clear()
        counts.clear()
        for center in current_centers:
            centers.add(center)
        for count in current_counts:
            counts.add(count)
        counts.add(new_count)

        # Retourner le point avec son cluster assigné
        yield (nearest_center_idx, point)

def create_data_with_keys():
    """Crée des données avec une clé unique pour tous les points"""
    data = []
    # Groupe 1
    for i in range(100):
        point = [5 + np.random.normal(0, 1), 5 + np.random.normal(0, 1)]
        data.append((0, point))  # 0 est la clé
    # Groupe 2
    for i in range(100):
        point = [10 + np.random.normal(0, 1), 10 + np.random.normal(0, 1)]
        data.append((0, point))  # même clé pour tous les points
    return data

def run_kmeans():
    with beam.Pipeline() as pipeline:
        # Paramètres
        k = 2

        # Création des données
        data = (pipeline
                | "Create data" >> beam.Create(create_data_with_keys())
                | "Process points" >> beam.ParDo(KMeansSequentialState(k))
                | "Print results" >> beam.Map(print))

if __name__ == '__main__':
    run_kmeans()

(0, array([4.41720167, 4.61768467]))
(0, array([5.67866317, 5.14389706]))
(0, array([6.77856565, 6.31982044]))
(0, array([4.62257359, 4.40450883]))
(0, array([5.64158601, 5.56571598]))
(0, array([3.51346764, 5.81685318]))
(0, array([4.77114499, 4.71417944]))
(0, array([5.21033151, 5.34744788]))
(0, array([3.27459727, 4.94640012]))
(0, array([5.18281875, 4.22835737]))
(0, array([5.15515701, 4.18673588]))
(0, array([3.58708385, 4.82929457]))
(0, array([6.05754298, 5.48358896]))
(0, array([5.69017219, 5.2642571 ]))
(0, array([5.92050172, 6.1194096 ]))
(0, array([4.00638658, 4.43633632]))
(0, array([3.67369044, 4.30092158]))
(0, array([4.92043514, 3.12846639]))
(0, array([5.78229607, 5.27410204]))
(0, array([4.8366794 , 5.42606892]))
(0, array([5.30570771, 5.34635853]))
(0, array([6.6407949 , 4.28506906]))
(0, array([6.10225032, 6.58972256]))
(0, array([5.84364087, 2.00015591]))
(0, array([5.53025224, 4.0067109 ]))
(0, array([5.91497532, 5.09776496]))
(0, array([5.37397042, 5.59914936]))
(

Problèmes avec une clé unique :


Tout le traitement se fait sur un seul worker

L'état est maintenu à un seul endroit

Pas de parallélisation réelle possible

Goulot d'étranglement potentiel pour de grands volumes de données

Solution éventuelle pour remedier à cela :

-  Utilisation de plusieurs clé .
chaque clé est traité par un workers differents , les etats sont distribué permettant un traitement paralléle.

-  Partitionnement par lot (batching).  
Regroupe les points en lots de taille fixe
Chaque lot reçoit une clé différente
Les lots peuvent être traités en parallèle

## Kmeans sequentiel distribué avec plusieurs clé

In [18]:
import apache_beam as beam
from apache_beam.transforms.userstate import BagStateSpec
import numpy as np

class KMeansSequentialState(beam.DoFn):
    # Définition des états pour stocker les centres et leurs effectifs
    centers_state = BagStateSpec('centers', beam.coders.PickleCoder())
    counts_state = BagStateSpec('counts', beam.coders.PickleCoder())

    def __init__(self, k: int):
        self.k = k

    def process(self, element, centers=beam.DoFn.StateParam(centers_state),
                counts=beam.DoFn.StateParam(counts_state)):
        # element est un tuple (key, point)
        key, point_data = element
        point = np.array([float(point_data[0]), float(point_data[1])])

        # Initialisation des centres si nécessaire
        current_centers = list(centers.read())
        current_counts = list(counts.read())

        if not current_centers:
            centers.add(point)
            counts.add(1)
            yield (key, (point, 0))  # 0 est l'index du cluster
            return

        if len(current_centers) < self.k:
            # Ajouter comme nouveau centre si on n'a pas encore k centres
            centers.add(point)
            counts.add(1)
            yield (key, (point, len(current_centers)))
            return

        # Trouver le centre le plus proche
        distances = [np.linalg.norm(point - center) for center in current_centers]
        nearest_center_idx = np.argmin(distances)

        # Mise à jour du compteur pour ce centre
        new_count = current_counts[nearest_center_idx] + 1

        # Mise à jour du centre
        current_centers[nearest_center_idx] = (
            current_centers[nearest_center_idx] +
            (1.0 / new_count) * (point - current_centers[nearest_center_idx])
        )

        # Mettre à jour les états
        centers.clear()
        counts.clear()
        for center in current_centers:
            centers.add(center)
        for i, count in enumerate(current_counts):
            counts.add(new_count if i == nearest_center_idx else count)

        # Yield le point avec son cluster assigné
        yield (key, (point, nearest_center_idx))

def create_data_with_two_keys():
    """Crée des données réparties sur deux clés"""
    data = []
    # Premier groupe de points (clé 0)
    for i in range(50):
        point = [5 + np.random.normal(0, 1), 5 + np.random.normal(0, 1)]
        data.append((0, point))
    for i in range(50):
        point = [10 + np.random.normal(0, 1), 10 + np.random.normal(0, 1)]
        data.append((0, point))

    # Deuxième groupe de points (clé 1)
    for i in range(50):
        point = [5 + np.random.normal(0, 1), 5 + np.random.normal(0, 1)]
        data.append((1, point))
    for i in range(50):
        point = [10 + np.random.normal(0, 1), 10 + np.random.normal(0, 1)]
        data.append((1, point))

    return data

class PrintResults(beam.DoFn):
    def process(self, element):
        key, (point, cluster) = element
        print(f"Key: {key}, Point: [{point[0]:.2f}, {point[1]:.2f}], Cluster: {cluster}")
        yield element

def run_kmeans():
    with beam.Pipeline() as pipeline:
        k = 2

        # Pipeline avec deux clés
        results = (pipeline
                  | "Create data" >> beam.Create(create_data_with_two_keys())
                  | "Process points" >> beam.ParDo(KMeansSequentialState(k))
                  | "Print results" >> beam.ParDo(PrintResults()))

if __name__ == '__main__':
    run_kmeans()

Key: 0, Point: [4.93, 4.24], Cluster: 0
Key: 0, Point: [5.23, 3.41], Cluster: 1
Key: 0, Point: [6.13, 4.67], Cluster: 0
Key: 0, Point: [4.54, 4.32], Cluster: 0
Key: 0, Point: [1.29, 4.91], Cluster: 0
Key: 0, Point: [6.97, 3.38], Cluster: 1
Key: 0, Point: [4.75, 4.55], Cluster: 0
Key: 0, Point: [6.59, 4.82], Cluster: 1
Key: 0, Point: [3.53, 4.48], Cluster: 0
Key: 0, Point: [4.36, 5.62], Cluster: 0
Key: 0, Point: [4.83, 5.45], Cluster: 0
Key: 0, Point: [3.42, 5.88], Cluster: 0
Key: 0, Point: [5.07, 5.13], Cluster: 0
Key: 0, Point: [8.16, 4.70], Cluster: 1
Key: 0, Point: [3.68, 4.78], Cluster: 0
Key: 0, Point: [4.16, 5.07], Cluster: 0
Key: 0, Point: [5.97, 4.89], Cluster: 1
Key: 0, Point: [3.53, 5.50], Cluster: 0
Key: 0, Point: [3.77, 3.87], Cluster: 0
Key: 0, Point: [4.03, 4.87], Cluster: 0
Key: 0, Point: [4.62, 4.04], Cluster: 0
Key: 0, Point: [5.77, 5.96], Cluster: 1
Key: 0, Point: [3.79, 5.62], Cluster: 0
Key: 0, Point: [7.11, 2.80], Cluster: 1
Key: 0, Point: [5.13, 4.45], Cluster: 0


## Conclusion sur l'implémentation de k-means séquentiel distribué (Partie D)

Cette implémentation combine l'approche séquentielle avec les capacités de distribution d'Apache Beam, explorant deux stratégies distinctes de parallélisation.

### Approche avec clé unique

- **Limitations identifiées** :
  - Centralisation du traitement sur un seul worker
  - État unique partagé limitant la parallélisation
  - Goulot d'étranglement potentiel sur les grands volumes
  - Conservation de la séquentialité stricte de l'algorithme

### Approche avec clés multiples

- **Améliorations apportées** :
  - Distribution du traitement sur plusieurs workers
  - États distribués permettant un traitement parallèle
  - Meilleure scalabilité horizontale
  - Réduction du goulot d'étranglement

### Gestion des états avec BagStateSpec

- Utilisation efficace des états pour stocker les centres et effectifs
- Mise à jour incrémentale des centres préservant l'approche séquentielle
- Synchronisation des états entre les différentes clés

### Compromis et implications

- **Pour la parallélisation** :
  - Équilibre entre fidélité à l'algorithme séquentiel et distribution
  - Impact du nombre de clés sur la distribution du traitement
  - Compromis entre granularité de la distribution et cohérence des résultats

- **Pour la mémoire** :
  - État maintenu par clé plutôt que globalement
  - Duplication potentielle des centres entre les différentes clés
  - Consommation mémoire proportionnelle au nombre de clés

Cette implémentation démontre l'importance des choix de partitionnement dans un contexte distribué, tout en préservant les propriétés de l'algorithme k-means séquentiel. Elle illustre parfaitement le compromis nécessaire entre parallélisation et cohérence des résultats dans un environnement distribué.

# E.Implémentation d’une version streaming et distribuée de k-mean (Apache Beam)

In [19]:
import apache_beam as beam
from apache_beam.transforms.userstate import BagStateSpec
import numpy as np
from sklearn.cluster import KMeans

class StreamingKMeansState(beam.DoFn):
    # État pour stocker les batches
    batches_state = BagStateSpec('batches', beam.coders.PickleCoder())

    def __init__(self, n_clusters: int, max_batches: int = 5, history_weight: float = 0.8):
        self.n_clusters = n_clusters
        self.max_batches = max_batches
        self.history_weight = history_weight

    def process(self, element, batches=beam.DoFn.StateParam(batches_state)):
        key, batch = element
        current_batch = np.array(batch)

        # Lire les batches existants
        stored_batches = list(batches.read())

        # Ajouter le nouveau batch
        stored_batches.append(current_batch)

        # Garder seulement les max_batches plus récents
        if len(stored_batches) > self.max_batches:
            stored_batches = stored_batches[-self.max_batches:]

        # Calculer les poids pour chaque batch
        # Le batch le plus récent a l'index 0
        weights = []
        for i in range(len(stored_batches)):
            batch_weight = self.history_weight ** i
            weights.extend([batch_weight] * len(stored_batches[-(i+1)]))

        # Préparer toutes les données
        X = np.vstack(stored_batches)
        sample_weights = np.array(weights)

        # Initialiser ou mettre à jour les centroïdes
        kmeans = KMeans(n_clusters=self.n_clusters, init='k-means++', n_init=1)
        kmeans.fit(X, sample_weight=sample_weights)

        # Sauvegarder les batches mis à jour
        batches.clear()
        for b in stored_batches:
            batches.add(b)

        # Retourner les résultats pour ce batch
        labels = kmeans.predict(current_batch)
        return [(key, (current_batch, labels, kmeans.cluster_centers_))]

def generate_batch(n_points: int, time_step: int = 0):
    """Génère un batch de données avec concept drift"""
    # Centre 1 se déplace vers la droite
    center1 = [5 + time_step * 0.3, 5]
    # Centre 2 se déplace vers le haut
    center2 = [10, 10 + time_step * 0.2]

    # Génération des points
    points1 = np.random.normal(loc=center1, scale=1.0, size=(n_points // 2, 2))
    points2 = np.random.normal(loc=center2, scale=1.0, size=(n_points // 2, 2))

    return np.vstack([points1, points2])

def format_results(element):
    """Formate les résultats pour l'affichage"""
    key, (batch, labels, centroids) = element
    return f"""
Batch key: {key}
Nombre de points: {len(batch)}
Centroids:
{centroids}
-------------------"""

def run_pipeline():
    with beam.Pipeline() as pipeline:
        # Paramètres
        n_clusters = 2
        n_batches = 10
        points_per_batch = 100
        max_batches_memory = 5
        history_weight = 0.8

        # Créer les batches avec concept drift
        batches = []
        for i in range(n_batches):
            batch = generate_batch(points_per_batch, i)
            # Alterner entre deux clés pour la parallélisation
            key = i % 2
            batches.append((key, batch))

        # Pipeline
        results = (pipeline
                  | "Create batches" >> beam.Create(batches)
                  | "Process batches" >> beam.ParDo(StreamingKMeansState(
                      n_clusters=n_clusters,
                      max_batches=max_batches_memory,
                      history_weight=history_weight))
                  | "Format results" >> beam.Map(format_results)
                  | "Print" >> beam.Map(print))

if __name__ == '__main__':
    run_pipeline()


Batch key: 0
Nombre de points: 100
Centroids:
[[ 4.70388686  5.08535468]
 [ 9.86489792 10.17416194]]
-------------------

Batch key: 1
Nombre de points: 100
Centroids:
[[ 5.40476297  4.89370459]
 [ 9.96617465 10.11821565]]
-------------------

Batch key: 0
Nombre de points: 100
Centroids:
[[ 5.04269296  4.97850913]
 [ 9.87119093 10.30071046]]
-------------------

Batch key: 1
Nombre de points: 100
Centroids:
[[ 5.50340997  4.91070937]
 [ 9.94338986 10.3209515 ]]
-------------------

Batch key: 0
Nombre de points: 100
Centroids:
[[ 5.36377886  5.00594909]
 [ 9.87249312 10.40863447]]
-------------------

Batch key: 1
Nombre de points: 100
Centroids:
[[ 5.77155054  4.89601823]
 [ 9.90314152 10.53938165]]
-------------------

Batch key: 0
Nombre de points: 100
Centroids:
[[ 5.62723249  5.00909945]
 [ 9.87004743 10.51764481]]
-------------------

Batch key: 1
Nombre de points: 100
Centroids:
[[ 9.89752025 10.66435342]
 [ 6.01920977  4.96255942]]
-------------------

Batch key: 0
Nombre de 

## Conclusion sur l'implémentation streaming et distribuée de k-means (Partie E)

Cette implémentation finale combine les avantages des approches streaming et distribuée, offrant une solution complète pour le traitement de grands volumes de données évoluant dans le temps. Analysons les aspects clés de cette implémentation :

### Fusion des approches précédentes

- **Héritage du streaming (Partie B)** :
  - Gestion des batches avec fenêtre glissante
  - Pondération temporelle des données
  - Adaptation au concept drift
  
- **Héritage du distribué (Parties C et D)** :
  - Utilisation des états distribués avec `BagStateSpec`
  - Parallélisation via le système de clés
  - Distribution du traitement sur plusieurs workers

### Innovations et améliorations

- **Gestion avancée des données** :
  - Combinaison efficace du streaming et de la distribution
  - Maintien des batches récents par clé
  - Pondération temporelle préservée dans un contexte distribué

- **Optimisations techniques** :
  - Utilisation de scikit-learn pour le clustering
  - Initialisation k-means++ pour une meilleure stabilité
  - Alternance des clés pour équilibrer la charge

### Forces de l'implémentation

- Capacité à gérer simultanément :
  - De grands volumes de données (aspect distribué)
  - L'évolution temporelle des données (aspect streaming)
  - La répartition de la charge (parallélisation)
  - L'adaptation aux changements de distribution

### Compromis et considérations

- **Performance vs Précision** :
  - Équilibre entre fraîcheur des données et utilisation mémoire
  - Compromise entre distribution et cohérence des clusters
  - Impact du nombre de clés sur la granularité du traitement

Cette implémentation représente une synthèse mature des différentes approches explorées dans le projet, offrant une solution complète et équilibrée pour le clustering de données en environnement Big Data.